In [1]:
#Carregando dados
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

df_orders = pd.read_csv(r'C:\GameStoreBrasil\output\orders_cleaned.csv', parse_dates=['order_date'] )
df_games = pd.read_csv(r'C:\GameStoreBrasil\data\games.csv', parse_dates=['release_date'])
df_member = pd.read_csv(r'C:\GameStoreBrasil\output\members_cleaned.csv', parse_dates=['join_date'])

In [2]:
print(f'linhas carregadas: {len(df_orders)}')
linhas_antes = len(df_orders)

linhas carregadas: 5000


In [3]:
#filtro em orders

df_orders = df_orders[(df_orders['quantity']>0 ) & (df_orders['unit_price']>0)]

In [4]:
linhas_removidas = linhas_antes - len(df_orders)
print(f'Linhas removidas {linhas_removidas}')

Linhas removidas 175


In [5]:
#preenchendo ausentes

df_orders['order_date'] = df_orders['order_date'].fillna('0')

In [6]:
df_orders['order_date'] = pd.to_datetime(df_orders['order_date'], errors='coerce')

In [7]:
df_orders['revenue'] = df_orders['quantity'] * df_orders["unit_price"]

In [8]:
#Agrupando por dias! 

df_orders_novo = df_orders.set_index('order_date').resample('D').agg(
    revenue_day = ('revenue', 'sum')
)

#Esse codigo de agrupar por dia e fazer a receita diaria é igual a esse do daily! so que o dele ele usa asfreq e eu nao! e ele ja cria ali o revenue 

In [9]:
daily = (
    df_orders
    .set_index("order_date")["revenue"]
    .resample("D")
    .sum()
    .asfreq("D", fill_value=0)
)

In [10]:
print("\n ---- Serie temporal Diaria ----")
print(f'Periodo : {daily.index.min().strftime('%Y-%m-%d')} a {daily.index.max().strftime('%Y-%m-%d')}')
print(f'Total de dias {len(daily)}\n')
print(f' dias com vendas{(daily>0).sum()} \n ')
print(f' Dias sem vendas (preenchidos com 0) {(daily==0).sum()} \n ')
print(f' Receita total do periodo: R$ {daily.sum():.2f} \n ')
print(f' Media diaria de receita no periodo: R$ {daily.mean():.2f} \n ')
print('\n primeiras 5 linhas')
print(daily.head())
print('\n ultimas 5 linhas')
print(daily.tail())


 ---- Serie temporal Diaria ----
Periodo : 2024-01-01 a 2025-06-29
Total de dias 546

 dias com vendas546 
 
 Dias sem vendas (preenchidos com 0) 0 
 
 Receita total do periodo: R$ 1758554.06 
 
 Media diaria de receita no periodo: R$ 3220.79 
 

 primeiras 5 linhas
order_date
2024-01-01    3271.97
2024-01-02    4357.09
2024-01-03    4507.61
2024-01-04    3236.44
2024-01-05    4147.41
Freq: D, Name: revenue, dtype: float64

 ultimas 5 linhas
order_date
2025-06-25    4755.73
2025-06-26    1617.40
2025-06-27    6517.56
2025-06-28    2525.74
2025-06-29    3511.64
Freq: D, Name: revenue, dtype: float64


In [11]:
print("\n ---- Serie temporal Diaria ----")
print(f'Periodo : {df_orders_novo.index.min().strftime('%Y-%m-%d')} a {df_orders_novo.index.max().strftime('%Y-%m-%d')}')
print(f'Total de dias {len(df_orders_novo)}\n')
print(f'\n dias com vendas{(df_orders_novo>0).sum()} \n ')
print(f'\n Dias sem vendas (preenchidos com 0) {(df_orders_novo==0).sum()} \n ')
print(f'\n Media diaria de receita no periodo: R$ {df_orders_novo.mean()} \n ')

print('\n primeiras 5 linhas')
print(df_orders_novo.head())
print('\n ultimas 5 linhas')
print(df_orders_novo.tail())


 ---- Serie temporal Diaria ----
Periodo : 2024-01-01 a 2025-06-29
Total de dias 546


 dias com vendasrevenue_day    546
dtype: int64 
 

 Dias sem vendas (preenchidos com 0) revenue_day    0
dtype: int64 
 

 Media diaria de receita no periodo: R$ revenue_day    3220.794982
dtype: float64 
 

 primeiras 5 linhas
            revenue_day
order_date             
2024-01-01      3271.97
2024-01-02      4357.09
2024-01-03      4507.61
2024-01-04      3236.44
2024-01-05      4147.41

 ultimas 5 linhas
            revenue_day
order_date             
2025-06-25      4755.73
2025-06-26      1617.40
2025-06-27      6517.56
2025-06-28      2525.74
2025-06-29      3511.64


In [12]:
#Visualizar a serie temporal (Grafico rapido de verificação)

fig,ax = plt.subplots(figsize = (12,4))
daily.plot(ax = ax, color = '#1f77b4', linewidth = 0.8)
ax.set_title('Receita Diaria da GameStoreBrasil', fontsize = 14, fontweight = 'bold')
ax.set_ylabel('Receita (R$)')
ax.set_xlabel("Data")
plt.tight_layout()
plt.savefig(r'C:\GameStoreBrasil\output\temp_daily_series.png', dpi = 150, bbox_inches = 'tight')
print("\n - Grafico da serie diaria salvo em: output\temp_daily_series.png")
plt.close(fig)


 - Grafico da serie diaria salvo em: output	emp_daily_series.png


In [17]:
dias_teste = 30
treino = daily.iloc[:-dias_teste]
teste = daily.iloc[-dias_teste:]
modelo = ARIMA(treino, order=(1,1,1))
resultado_fit = modelo.fit()
previsao_teste = resultado_fit.forecast(steps = 30)
mae = mean_absolute_error(teste, previsao_teste)

print(f'\n - Total de dias na serie original: {len(daily)}')
print(f'\n - Dias no conjunto de TREINO: {len(treino)}')
print(f'\n - Dias no conjunto de TESTE: {len(teste)}')
print(f'\n - Periodo do teste (a "prova"): {teste.index.min().strftime('%Y-%m-%d')} a {teste.index.max().strftime('%Y-%m-%d')}')
print(modelo)


 - Total de dias na serie original: 546

 - Dias no conjunto de TREINO: 516

 - Dias no conjunto de TESTE: 30

 - Periodo do teste (a "prova"): 2025-05-31 a 2025-06-29


In [18]:
fig,ax = plt.subplots(figsize = (12,4))

treino.plot(ax=ax, color = '#1f77b4', label = 'Conjunto de Treino')
treino.plot(ax=ax, color = '#d62728', label = 'Conjunto de Teste ( Validação )')

ax.set_title("Divisão da serie temporal: Treino vs Teste", fontsize = 14, fontweight = 'bold')
ax.set_ylabel('Receita Diaria (R$)')
ax.legend()

plt.tight_layout()
plt.savefig(r"C:\GameStoreBrasil\output\temp_train_test_split.png")
plt.close(fig)

In [19]:
teste = daily.iloc[-dias_teste:]
modelo = ARIMA(treino, order=(1,1,1))
resultado_fit = modelo.fit()
previsao_teste = resultado_fit.forecast(steps = 30)
mae = mean_absolute_error(teste, previsao_teste)

print(f'MAE ( Erro Medio ABSOLUTO) no conjunto de teste: R${mae:.2f}')
print(f'Interpretação em media, o modelo erra a previsão diaria em R$ {mae:.2f}')
print(f'\n AIC do modelo: {resultado_fit.aic:.2f}')

MAE ( Erro Medio ABSOLUTO) no conjunto de teste: R$1189.21
Interpretação em media, o modelo erra a previsão diaria em R$ 1189.21

 AIC do modelo: 8815.92


In [20]:
print('Ajustando Arima(1,1,1) em TODA a serie historica')
modelo_final = ARIMA(daily, order=(1,1,1))
resultado_final = modelo_final.fit()

previsao_futura = resultado_final.forecast(steps=30)

ultima_data = daily.index.max()
datas_futuras = pd.date_range(start = ultima_data + pd.Timedelta(days=1), periods = 30, freq = 'D')


df_forecast = pd.DataFrame({
    'Data' : datas_futuras.strftime('%d-%m-%Y'),
    'Predicted_Sales' : previsao_futura.values.round(2)
})



Ajustando Arima(1,1,1) em TODA a serie historica


In [21]:
print('\n ---- Primeiras 5 linhas da previsao final ----')

print(df_forecast.head())

print('\n ultimas 5 linhas da previsao final')

print(df_forecast.tail())



 ---- Primeiras 5 linhas da previsao final ----
         Data  Predicted_Sales
0  30-06-2025          3186.96
1  01-07-2025          3220.63
2  02-07-2025          3217.14
3  03-07-2025          3217.50
4  04-07-2025          3217.47

 ultimas 5 linhas da previsao final
          Data  Predicted_Sales
25  25-07-2025          3217.47
26  26-07-2025          3217.47
27  27-07-2025          3217.47
28  28-07-2025          3217.47
29  29-07-2025          3217.47


In [22]:
caminho_csv = r"C:\GameStoreBrasil\output\Session1_SalesForecast.csv"

df_forecast.to_csv(caminho_csv, index=False)

print(f" - Previsao Salva com sucesso em: {caminho_csv}")
print(" - TAREFA 1.6 CONCLUIDA COM SUCESSO - ")

 - Previsao Salva com sucesso em: C:\GameStoreBrasil\output\Session1_SalesForecast.csv
 - TAREFA 1.6 CONCLUIDA COM SUCESSO - 


In [ ]:
resultado = adfuller(daily)
print("p-value:", resultado[1]) # p > 0.05 sugere não-estacionária -> usar d=1